# Chương 3: Mạng Nơ-ron Hồi Quy - IMDB và Sine Wave

## Mục tiêu
Phân tích khả năng ghi nhớ chuỗi với IMDB Sentiment và Sine Wave.

## Datasets
1. IMDB Sentiment: Text classification
2. Sine Wave: Simple sequence prediction

In [13]:
# Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create directory for saved plots
import os
if not os.path.exists('plots'):
    os.makedirs('plots')

print("RNN Chapter - Plots will be saved to 'plots' directory")

RNN Chapter - Plots will be saved to 'plots' directory


## Dataset 1: IMDB Sentiment

### Load và Visualize Data

In [14]:
# Load IMDB Dataset
max_features = 10000
maxlen = 500

(x_train_imdb, y_train_imdb), (x_test_imdb, y_test_imdb) = keras.datasets.imdb.load_data(num_words=max_features)

print("IMDB Dataset:")
print(f"Training data shape: {len(x_train_imdb)}")
print(f"Test data shape: {len(x_test_imdb)}")
print(f"Average review length: {np.mean([len(x) for x in x_train_imdb]):.0f} words")
print(f"Max review length: {max([len(x) for x in x_train_imdb])} words")

# Pad sequences
x_train_imdb = keras.preprocessing.sequence.pad_sequences(x_train_imdb, maxlen=maxlen)
x_test_imdb = keras.preprocessing.sequence.pad_sequences(x_test_imdb, maxlen=maxlen)

print(f"Padded training data shape: {x_train_imdb.shape}")

IMDB Dataset:
Training data shape: 25000
Test data shape: 25000
Average review length: 239 words
Max review length: 2494 words
Padded training data shape: (25000, 500)


In [15]:
# Plot 1: Class Distribution
plt.figure(figsize=(8, 6))
unique, counts = np.unique(y_train_imdb, return_counts=True)
plt.bar(['Negative', 'Positive'], counts)
plt.title('Class Distribution in IMDB Training Set')
plt.ylabel('Count')
plt.savefig('plots/imdb_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: Review Length Distribution
review_lengths = [len(x) for x in x_train_imdb]
plt.figure(figsize=(10, 6))
plt.hist(review_lengths, bins=50, alpha=0.7)
plt.title('Review Length Distribution')
plt.xlabel('Review Length')
plt.ylabel('Frequency')
plt.savefig('plots/imdb_review_length_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: Word Frequency (Top 20)
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Count word frequencies
from collections import Counter
all_words = []
for review in x_train_imdb[:1000]:  # Sample for speed
    for word_id in review:
        if word_id > 3:  # Skip special tokens
            all_words.append(reverse_word_index.get(word_id, '?'))

word_freq = Counter(all_words)
top_words = word_freq.most_common(20)

plt.figure(figsize=(12, 8))
words, counts = zip(*top_words)
plt.barh(words[::-1], counts[::-1])
plt.title('Top 20 Most Frequent Words in IMDB Reviews')
plt.xlabel('Frequency')
plt.savefig('plots/imdb_top_words.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 4: Sentiment by Review Length
plt.figure(figsize=(10, 6))
positive_lengths = [len(x) for x, y in zip(x_train_imdb, y_train_imdb) if y == 1]
negative_lengths = [len(x) for x, y in zip(x_train_imdb, y_train_imdb) if y == 0]

plt.hist(positive_lengths, bins=50, alpha=0.5, label='Positive', density=True)
plt.hist(negative_lengths, bins=50, alpha=0.5, label='Negative', density=True)
plt.title('Review Length Distribution by Sentiment')
plt.xlabel('Review Length')
plt.ylabel('Density')
plt.legend()
plt.savefig('plots/imdb_sentiment_by_length.png', dpi=300, bbox_inches='tight')
plt.show()

### Build và Train RNN Model

In [16]:
# Build Simple RNN Model for IMDB
model_imdb = keras.Sequential([
    layers.Embedding(max_features, 32, input_length=maxlen),
    layers.SimpleRNN(32),
    layers.Dense(1, activation='sigmoid')
])

model_imdb.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])

model_imdb.summary()

# Train the model
history_imdb = model_imdb.fit(x_train_imdb, y_train_imdb,
                              epochs=5, batch_size=128,
                              validation_split=0.2, verbose=1)

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_imdb.history['accuracy'], label='Training Accuracy')
plt.plot(history_imdb.history['val_accuracy'], label='Validation Accuracy')
plt.title('IMDB - RNN Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_imdb.history['loss'], label='Training Loss')
plt.plot(history_imdb.history['val_loss'], label='Validation Loss')
plt.title('IMDB - RNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('plots/imdb_rnn_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Evaluate
test_loss, test_acc = model_imdb.evaluate(x_test_imdb, y_test_imdb, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")

# Confusion Matrix
y_pred_imdb = (model_imdb.predict(x_test_imdb) > 0.5).astype(int).flatten()
cm = confusion_matrix(y_test_imdb, y_pred_imdb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.title('IMDB - RNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('plots/imdb_rnn_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_4 (Embedding)     (None, 500, 32)           320000    
                                                                 
 simple_rnn_4 (SimpleRNN)    (None, 32)                2080      
                                                                 
 dense_4 (Dense)             (None, 1)                 33        
                                                                 
Total params: 322113 (1.23 MB)
Trainable params: 322113 (1.23 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/5


157/157 [==============================] - 6s 34ms/step - loss: 0.6593 - accuracy: 0.5908 - val_loss: 0.6288 - val_accuracy: 0.6682
Epoch 2/5
157/157 [==============================] - 5s 33ms/step - loss: 0.4346 - accuracy: 0.8072 - val_loss: 0.4580 - val_accuracy: 0.7914
Epoch 3/5
157/157 [==============================] - 5s 32ms/step - loss: 0.2978 - accuracy: 0.8795 - val_loss: 0.3886 - val_accuracy: 0.8344
Epoch 4/5
157/157 [==============================] - 5s 34ms/step - loss: 0.1733 - accuracy: 0.9369 - val_loss: 0.4317 - val_accuracy: 0.8182
Epoch 5/5
157/157 [==============================] - 5s 32ms/step - loss: 0.0884 - accuracy: 0.9730 - val_loss: 0.4760 - val_accuracy: 0.8350
Test Accuracy: 0.8409
782/782 [==============================] - 4s 5ms/step


In [17]:
# === Model 2: Bidirectional RNN (IMDB) ===
from tensorflow.keras.layers import Bidirectional
model_imdb_2 = keras.Sequential([
    keras.layers.Embedding(max_features, 32),
    Bidirectional(keras.layers.SimpleRNN(32)),
    keras.layers.Dense(1, activation='sigmoid')
])
model_imdb_2.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
history_imdb_2 = model_imdb_2.fit(x_train_imdb, y_train_imdb,
                                   epochs=5, batch_size=128,
                                   validation_split=0.2, verbose=0)
print("BiRNN IMDB val_acc:", history_imdb_2.history['val_accuracy'][-1])

# Plot 1: Loss/Accuracy comparison (IMDB)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_imdb.history['accuracy'],   label='Vanilla RNN')
axes[0].plot(history_imdb_2.history['accuracy'], label='BiRNN')
axes[0].set_title('IMDB Training Accuracy'); axes[0].legend()
axes[1].plot(history_imdb.history['loss'],   label='Vanilla RNN')
axes[1].plot(history_imdb_2.history['loss'], label='BiRNN')
axes[1].set_title('IMDB Training Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('plots/ch3_imdb_history.png', dpi=300)
plt.close()
print("Saved: plots/ch3_imdb_history.png")


BiRNN IMDB val_acc: 0.8582000136375427
Saved: plots/ch3_imdb_history.png


In [18]:

def predict_sentiment(text):
    text = text.lower().replace('<br />', ' ')
    tokens = text.split()
    seq = [word_index.get(w, 2) for w in tokens]
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    padded = pad_sequences([seq], maxlen=maxlen)
    pred = model_imdb.predict(padded)[0][0]
    return "Positive" if pred > 0.5 else "Negative", pred

sample_reviews = ["This movie was wonderful!", "I hated this film."]
results = [predict_sentiment(r) for r in sample_reviews]

plt.figure(figsize=(8, 4))
plt.barh(['Review 1', 'Review 2'], [r[1] for r in results], color=['green', 'red'])
plt.xlim(0, 1)
plt.title('Sample Prediction Scores (IMDB)')
plt.axvline(x=0.5, color='black', linestyle='--')
plt.savefig('plots/ch3_imdb_prediction.png', dpi=300)
plt.show()


1/1 [==============================] - 0s 16ms/step


In [19]:

def predict_sentiment(text):
    text = text.lower().replace('<br />', ' ')
    tokens = text.split()
    seq = [word_index.get(w, 2) for w in tokens]
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    padded = pad_sequences([seq], maxlen=maxlen)
    pred = model_imdb.predict(padded)[0][0]
    return "Positive" if pred > 0.5 else "Negative", pred

sample_reviews = ["This movie was wonderful!", "I hated this film."]
results = [predict_sentiment(r) for r in sample_reviews]

plt.figure(figsize=(8, 4))
plt.barh(['Review 1', 'Review 2'], [r[1] for r in results], color=['green', 'red'])
plt.xlim(0, 1)
plt.title('Sample Prediction Scores (IMDB)')
plt.axvline(x=0.5, color='black', linestyle='--')
plt.savefig('plots/ch3_imdb_prediction.png', dpi=300)
plt.show()


1/1 [==============================] - 0s 15ms/step


## Dataset 2: Sine Wave (Sequence Prediction)

### Generate và Visualize Data

In [20]:
# Generate Sine Wave Dataset
def generate_sine_wave(seq_length=1000, look_back=10):
    # Generate sine wave with noise
    t = np.linspace(0, 4*np.pi, seq_length)
    data = np.sin(t) + 0.1 * np.random.randn(seq_length)

    # Create sequences
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i+look_back])
        y.append(data[i+look_back])

    X = np.array(X)
    y = np.array(y)

    # Reshape for RNN
    X = X.reshape((X.shape[0], X.shape[1], 1))

    return X, y, data

X_sine, y_sine, raw_data = generate_sine_wave()

# Split data
train_size = int(len(X_sine) * 0.8)
X_train_sine, X_test_sine = X_sine[:train_size], X_sine[train_size:]
y_train_sine, y_test_sine = y_sine[:train_size], y_sine[train_size:]

print("Sine Wave Dataset:")
print(f"Training data shape: {X_train_sine.shape}")
print(f"Test data shape: {X_test_sine.shape}")
print(f"Sequence length: {X_train_sine.shape[1]}")

Sine Wave Dataset:
Training data shape: (792, 10, 1)
Test data shape: (198, 10, 1)
Sequence length: 10


In [21]:
# Plot 1: Raw Sine Wave Data
plt.figure(figsize=(15, 6))
plt.plot(raw_data[:200], label='Raw Data')
plt.title('Sine Wave Time Series (First 200 points)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.savefig('plots/sine_wave_raw_data.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: Sample Sequences
plt.figure(figsize=(12, 8))
for i in range(5):
    plt.subplot(5, 1, i+1)
    plt.plot(X_sine[i*100].flatten(), 'b-', label='Input Sequence')
    plt.plot([len(X_sine[i*100]), len(X_sine[i*100])], [X_sine[i*100][-1, 0], y_sine[i*100]], 'r--', label='Target')
    plt.title(f'Sample Sequence {i+1}')
    plt.legend()
plt.tight_layout()
plt.savefig('plots/sine_wave_sample_sequences.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: Data Distribution
plt.figure(figsize=(10, 6))
plt.hist(y_sine, bins=50, alpha=0.7)
plt.title('Distribution of Target Values')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.savefig('plots/sine_wave_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 4: Autocorrelation
from pandas.plotting import autocorrelation_plot
import pandas as pd

plt.figure(figsize=(10, 6))
autocorrelation_plot(pd.Series(raw_data[:500]))
plt.title('Autocorrelation Plot')
plt.savefig('plots/sine_wave_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

### Build và Train RNN Model for Sine Wave

In [22]:
# Build RNN Model for Sine Wave Prediction
model_sine = keras.Sequential([
    layers.SimpleRNN(50, input_shape=(X_train_sine.shape[1], 1)),
    layers.Dense(1)
])

model_sine.compile(optimizer='adam', loss='mse', metrics=['mae'])

model_sine.summary()

# Train the model
history_sine = model_sine.fit(X_train_sine, y_train_sine,
                              epochs=20, batch_size=32,
                              validation_split=0.2, verbose=1)

# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_sine.history['mae'], label='Training MAE')
plt.plot(history_sine.history['val_mae'], label='Validation MAE')
plt.title('Sine Wave - RNN Model MAE')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_sine.history['loss'], label='Training Loss')
plt.plot(history_sine.history['val_loss'], label='Validation Loss')
plt.title('Sine Wave - RNN Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('plots/sine_wave_rnn_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

# Predictions and visualization
y_pred_sine = model_sine.predict(X_test_sine).flatten()

plt.figure(figsize=(15, 6))
plt.plot(y_test_sine[:200], label='True Values', alpha=0.7)
plt.plot(y_pred_sine[:200], label='Predictions', alpha=0.7)
plt.title('Sine Wave Prediction Results')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.savefig('plots/sine_wave_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(y_test_sine, y_pred_sine)
mae = mean_absolute_error(y_test_sine, y_pred_sine)
print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 simple_rnn_6 (SimpleRNN)    (None, 50)                2600      
                                                                 
 dense_6 (Dense)             (None, 1)                 51        
                                                                 
Total params: 2651 (10.36 KB)
Trainable params: 2651 (10.36 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/20


20/20 [==============================] - 0s 6ms/step - loss: 0.3298 - mae: 0.4151 - val_loss: 0.0339 - val_mae: 0.1500
Epoch 2/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0323 - mae: 0.1461 - val_loss: 0.0405 - val_mae: 0.1655
Epoch 3/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0198 - mae: 0.1130 - val_loss: 0.0223 - val_mae: 0.1203
Epoch 4/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0170 - mae: 0.1054 - val_loss: 0.0217 - val_mae: 0.1193
Epoch 5/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0156 - mae: 0.1005 - val_loss: 0.0193 - val_mae: 0.1129
Epoch 6/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0147 - mae: 0.0982 - val_loss: 0.0180 - val_mae: 0.1087
Epoch 7/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0145 - mae: 0.0975 - val_loss: 0.0175 - val_mae: 0.1069
Epoch 8/20
20/20 [==============================] - 0s 2ms/step - loss: 0.0136 - mae: 0.0940 - val_lo

In [23]:
# === Model 2: Deep RNN (Sine Wave) ===
model_sine_2 = keras.Sequential([
    keras.layers.SimpleRNN(32, input_shape=(X_train_sine.shape[1], X_train_sine.shape[2]),
                           return_sequences=True),
    keras.layers.SimpleRNN(32),
    keras.layers.Dense(1)
])
model_sine_2.compile(optimizer='adam', loss='mse')
history_sine_2 = model_sine_2.fit(X_train_sine, y_train_sine, epochs=10,
                                   batch_size=16, verbose=0)
print("Deep RNN Sine MSE:", history_sine_2.history['loss'][-1])

# Plot 4: Loss comparison (Sine Wave)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(history_sine.history['loss'],   label='Vanilla RNN')
plt.plot(history_sine_2.history['loss'], label='Deep RNN')
plt.title('Sine Wave Training Loss Comparison')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend()
plt.tight_layout()
plt.savefig('plots/ch3_sine_history.png', dpi=300)
plt.close()
print("Saved: plots/ch3_sine_history.png")


Deep RNN Sine MSE: 0.012920818291604519
Saved: plots/ch3_sine_history.png


/tmp/ipykernel_3890047/1194676419.py:17: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(8, 4))


In [24]:
# Move all saved PNG files to plots directory
import shutil
png_files = [f for f in os.listdir('.') if f.endswith('.png')]
for file in png_files:
    shutil.move(file, os.path.join('plots', file))

print("All RNN plots saved to 'plots' directory for report writing.")

All RNN plots saved to 'plots' directory for report writing.
